
# 03 执行算法与微观结构：盘口、下单与成本

> **本章要解决的问题**：回测里按收盘价成交、想买多少买多少，实盘为什么做不到？一张订单簿里到底藏着哪些成本？TWAP、VWAP、POV 差在哪，为什么同一笔母单的成交均价会不一样？

## 本章导读

前面几本教程讲的是“买什么”；这一章讲“怎么买”。同一笔观点，执行得好与坏，可能差出几个点的年化收益。

按“从盘口到对账”的顺序展开：

1. 订单簿：价差、深度、队列；
2. 订单类型与算法：市价、限价、TWAP、VWAP、POV；
3. 冲击成本与滑点：成本从哪来、怎么建模；
4. 成交率、排队与逆向选择：挂单为什么成交了反而亏；
5. 容量与执行对账：能装多少钱、实际和计划差在哪；
6. 从回测到实盘：执行假设清单。

读的时候记住一句话：**回测里的成交价是假设，执行里的成交价是结果。** 本章数据全部是合成盘口/合成行情（教学数据），参数是教学示例，不是真实交易所数据。

### 学习目标

- **任务 1**：读一张订单簿，算出价差、深度和订单失衡。
- **任务 2**：用逐档吃量算大单的成交均价与冲击，并与中间价比较。
- **任务 3**：为同一笔母单设计 TWAP/VWAP/POV，比较均价、滑点、未成交与机会成本。
- **任务 4**：用平方根模型估算冲击成本，并算出给定成本预算下的规模上限。
- **任务 5**：模拟限价单的成交率与 markout，解释逆向选择。
- **任务 6**：做一张执行对账表，把滑点拆成价差、冲击、时机与残差。

### 先修

第 31 章订单容量与可实现收益；第 14 章原生回放对账；会 numpy/pandas 即可。

---

## 在量化研究中的位置

- **谁用**：交易/执行研究员、组合经理、做市/自营、风控。
- **输入**：盘口快照/逐笔、母单计划、交易成本参数、账户约束。
- **输出**：下单计划、成交均价与滑点、成本估计、容量上限、执行对账。
- **在流程的哪一步**：信号 → 组合 → **执行** → 回放对账；本章是“组合权重”之后、账户盈亏之前的最后一公里。
- **失败模式**：按收盘价成交、忽略价差与冲击、用无限流动性回测、把未成交当成交、对账只对数量不对价格。
- **最小工作流**：看盘口 → 定母单 → 选算法 → 估成本 → 拆单执行 → 对账归因。

---

### 环境准备



In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



---

## 1. 订单簿：价差、深度、队列

### 1.1 限价订单簿的组成

订单簿按价格排两列：**买盘（bid）**从高到低，**卖盘（ask）**从低到高。每一档记价格和挂单量，最靠近中间价的一档叫一档（best bid / best ask）。中间价是最优买卖价的平均：

$$\text{mid}=\frac{\text{best bid}+\text{best ask}}{2}.$$

### 1.2 三个微观结构指标

- **价差（spread）** = best ask − best bid：立刻成交要付的“过路费”；
- **深度（depth）**：某档位后面还有多少量，决定大单能吃掉几档；
- **订单失衡（OFI）**：买盘量与卖盘量之差，常被用来做短周期方向预测。

价差越窄、深度越厚，市场越“好”；但深度是可以撤的，看到的量不等于能成交的量。\n\n价差和深度是静态快照，OFI 是动态信号：买方挂单突然变厚，说明有人急着买，短周期价格可能上行。但 OFI 只在极短周期有效，而且对撤单非常敏感。

### 1.3 代码：逐档吃量



In [2]:

import numpy as np

rng = np.random.default_rng(5301)

mid, tick, levels = 100.00, 0.01, 5
spread_ticks = 2
bids = mid - spread_ticks / 2 * tick - np.arange(levels) * tick
asks = mid + spread_ticks / 2 * tick + np.arange(levels) * tick
bid_size = rng.integers(200, 1200, levels)
ask_size = rng.integers(200, 1200, levels)

print("卖盘（由高到低）")
for p, q in zip(asks[::-1], ask_size[::-1]):
    print(f"  {p:.2f}  {q:>5}")
print(f"中间价 {mid:.2f}，价差 {asks[0] - bids[0]:.2f} 元 = {spread_ticks} 跳")
print("买盘（由高到低）")
for p, q in zip(bids, bid_size):
    print(f"  {p:.2f}  {q:>5}")


def walk_book(order_qty, side="buy"):
    book = asks if side == "buy" else bids
    sizes = ask_size if side == "buy" else bid_size
    remain, cost, filled = order_qty, 0.0, 0.0
    for p, q in zip(book, sizes):
        take = min(remain, q)
        cost += take * p
        filled += take
        remain -= take
        if remain <= 0:
            break
    return (cost / filled if filled > 0 else float("nan")), filled, remain


half_spread_bps = (spread_ticks / 2 * tick) / mid * 1e4
print(f"{'买入量':>6}{'成交均价':>10}{'总滑点bp':>10}{'未成交':>8}{'冲击部分bp':>11}")
for qty in [300, 1500, 4000, 8000]:
    avg, filled, left = walk_book(qty)
    slip = (avg - mid) / mid * 1e4
    print(f"{qty:>6}{avg:>10.4f}{slip:>10.1f}{left:>8.0f}{slip - half_spread_bps:>11.1f}")


卖盘（由高到低）
  100.05    947
  100.04    624
  100.03   1023
  100.02   1011
  100.01    899
中间价 100.00，价差 0.02 元 = 2 跳
买盘（由高到低）
  99.99    403
  99.98    749
  99.97   1019
  99.96    486
  99.95    912
   买入量      成交均价     总滑点bp     未成交     冲击部分bp
   300  100.0100       1.0       0        0.0
  1500  100.0140       1.4       0        0.4
  4000  100.0268       2.7       0        1.7
  8000  100.0294       2.9    3496        1.9



**读数与应用**：小单（300 手）只吃一档，滑点约等于半个价差（1bp）；随着单量变大，逐档吃上去，成交均价越来越差，4000 手的冲击部分开始超过价差。更早出现的限制是深度：8000 手有近 3500 手未成交——**“盘口深度”决定单笔能成交多少，而不是你想成交多少。**

**常见错误**：

1. 用中间价算成交，忽略半个价差。
2. 假设盘口深度无限，大单直接一笔成交。
3. 用快照挂单量当真实可成交量，忘了会被撤单。

**停下来想一想**：盘口上看到买盘很厚，是不是说明“支撑很强”？

<details><summary>对照你的思路</summary>

不一定。挂单可以随时撤，冰山单和隐藏单根本看不到；真实的承接能力要看成交、撤单率和价格反应。厚买盘也可能是“假墙”——价格一靠近就撤。盘口深度是“意愿”，成交才是“事实”。

</details>

---

## 2. 订单类型与执行算法

### 2.1 三类订单

| 订单 | 特点 | 代价 |
|---|---|---|
| 市价单 | 立刻成交、确定性高 | 付价差 + 冲击，大单很贵 |
| 限价单 | 价格可控、可能省钱 | 可能不成交，成交时可能逆向选择 |
| 算法单 | 拆成小单按规则执行 | 有跟踪误差、未成交风险 |

### 2.2 四种常见执行算法

- **TWAP**（时间加权）：按时间等分，最简单、可解释；
- **VWAP**（成交量加权）：按历史成交量分布拆单，贴近市场平均成本；
- **POV**（参与率）：每段时间不超过市场成交量的固定比例，控制冲击；
- **IS**（Implementation Shortfall）：在冲击与风险之间找平衡，通常按“越难成交越激进”调整。

算法不是越复杂越好：**没有成交量分布数据时，TWAP 是最稳的基线。**\n\nIS 的核心是把“冲击成本”和“未成交风险”放进同一个目标：拖得越久，价格可能跑得越远（时机风险）；追得越急，冲击越大。实际执行常把 IS 拆成激进、中性、被动三档，按信号强度和盘口状态切换。

### 2.3 代码：同一笔母单的三种执行



In [3]:

import numpy as np

rng = np.random.default_rng(5302)
n = 240                                            # 240 分钟
u = np.linspace(0, 1, n)
shape = 800 + 2200 * (np.exp(-((u - 0.0) ** 2) / 0.02)
                      + np.exp(-((u - 1.0) ** 2) / 0.02))
volume = shape + rng.integers(0, 200, n)
volume = volume / volume.sum() * 300000            # 全天 30 万手（教学数据）
price = 100 + np.cumsum(rng.normal(0, 0.02, n))
parent, arrival = 30000.0, price[0]


def summarize(fill_qty, label):
    filled = fill_qty.sum()
    avg = (fill_qty * price).sum() / filled
    slip = (avg - arrival) / arrival * 1e4
    remain = parent - filled
    opp = (remain / parent) * (price[-1] - arrival) / arrival * 1e4
    print(f"{label:>6}: 成交 {filled:>7.0f}，均价 {avg:.4f}，"
          f"滑点 {slip:6.1f}bp，未成交 {remain:>6.0f}，机会成本 {opp:6.1f}bp")


summarize(np.full(n, parent / n), "TWAP")                 # 按时间等分
summarize(volume / volume.sum() * parent, "VWAP")         # 按成交量分布

pov, remain = np.zeros(n), parent                         # 每分钟不超过 5% 成交量
for t in range(n):
    take = min(0.05 * volume[t], remain)
    pov[t] = take
    remain -= take
    if remain <= 0:
        break
summarize(pov, "POV5%")
print(f"到达价 {arrival:.4f}，收盘价 {price[-1]:.4f}，全天成交量 {volume.sum():.0f} 手")


  TWAP: 成交   30000，均价 99.9446，滑点   -6.2bp，未成交      0，机会成本   -0.0bp
  VWAP: 成交   30000，均价 99.9496，滑点   -5.7bp，未成交     -0，机会成本    0.0bp
 POV5%: 成交   15000，均价 99.9496，滑点   -5.7bp，未成交  15000，机会成本  -11.0bp
到达价 100.0064，收盘价 99.7855，全天成交量 300000 手



**读数与应用**：TWAP 与 VWAP 都完成了母单，但均价不同——VWAP 跟随成交量分布，开盘/收盘成交多，均价更接近“市场平均成本”。POV 5% 受参与率上限约束，全天最多做到 15000 手，一半没成交，机会成本要单独记；它的成交均价与 VWAP 相同，因为 5% 的参与率恰好等于按量比例拆分。另外本例价格全天从 100.01 跌到 99.79，晚买反而便宜，所以滑点为负——**执行比较必须在同一天、同一到达价下比，不能跨日比。**

**常见错误**：

1. 只看均价，不看未成交量与机会成本。
2. 把 VWAP 当“最优”：VWAP 只是跟随市场平均，不保证比 TWAP 便宜。
3. 用当天收盘后的成交量分布指导当天开盘的执行（未来函数）。

---

## 3. 冲击成本与滑点

### 3.1 成本的四块

一笔执行的总成本可以拆成：

$$\text{总成本}=\underbrace{\frac{\text{spread}}{2}}_{\text{价差}}+\underbrace{\text{impact}}_{\text{冲击}}+\underbrace{\text{timing}}_{\text{时机}}+\underbrace{\text{opportunity}}_{\text{未成交机会}}.$$

价差是“立刻成交的过路费”；冲击是“吃穿盘口、被别人看到”的代价；时机是执行期间价格漂移；机会成本是没成交的部分错过的行情。经验上，价差和冲击是“确定要被吃掉”的成本，时机成本取决于你等多久，机会成本取决于你能不能成交。

### 3.2 平方根冲击模型

行业里常用的经验形式是

$$\text{impact}\approx c\,\sigma\sqrt{\frac{Q}{V}},$$

其中 $\sigma$ 是波动率、$Q$ 是下单量、$V$ 是市场成交量、$c$ 是需按品种校准的系数（教学示例取 0.6）。它比“成本与规模成正比”更符合数据：小单成本由价差主导，大单成本由冲击主导。

### 3.3 代码：冲击成本与容量



In [4]:

import numpy as np

price, adv, sigma = 100.0, 300000.0, 0.02      # 教学示例：价 100、日成交量 30 万手、日波动 2%
half_spread_bps, c = 1.0, 0.6                  # 教学示例系数，需按品种校准


def cost_bps(q, model="sqrt"):
    s = q / adv
    if model == "sqrt":
        impact = c * sigma * np.sqrt(s) * 1e4
    else:                                       # 线性模型：成本与规模成正比
        impact = (c * sigma * np.sqrt(0.10) * 1e4 / 0.10) * s
    return half_spread_bps + impact


print(f"{'下单量':>8}{'参与率':>8}{'平方根bp':>10}{'线性bp':>9}{'差异':>9}")
for q in [3000, 10000, 30000, 100000]:
    sq, ln = cost_bps(q), cost_bps(q, "linear")
    print(f"{q:>8}{q / adv:>8.1%}{sq:>10.1f}{ln:>9.1f}{sq - ln:>9.1f}")

budget = 30.0
q_max = adv * ((budget - half_spread_bps) / (c * sigma * 1e4)) ** 2
print(f"成本预算 {budget:.0f}bp 下的规模上限约 {q_max:,.0f} 手（{q_max / adv:.1%} ADV）")
print(f"若只用线性模型，会算出 {adv * (budget - half_spread_bps) / (c * sigma * np.sqrt(0.10) * 1e4 / 0.10):,.0f} 手")


     下单量     参与率     平方根bp     线性bp       差异
    3000    1.0%      13.0      4.8      8.2
   10000    3.3%      22.9     13.6      9.3
   30000   10.0%      38.9     38.9      0.0
  100000   33.3%      70.3    127.5    -57.2
成本预算 30bp 下的规模上限约 17,521 手（5.8% ADV）
若只用线性模型，会算出 22,927 手



**读数与应用**：参与率从 1% 升到 33%，平方根成本从约 13bp 升到约 70bp，但**增长是次线性的**——这就是平方根模型：规模翻四倍，冲击只翻约两倍。线性模型会低估小单（1% 时只算 4.8bp，实际价差就要 1bp 再加冲击），却高估大单。按 30bp 成本预算反解，平方根模型给出约 1.75 万手（5.8% ADV），线性模型却给出约 2.29 万手、乐观了 31%——**容量不是拍脑袋，是用成本预算反推出来的。**

**常见错误**：

1. 成本与规模成正比，容量算得过大或过小。
2. 忽略价差，只算冲击。
3. 用全市场成交量当“可参与量”，忘了自己的单子就是市场成交量的一部分。

---

## 4. 成交率、排队与逆向选择

### 4.1 限价单的成交率

限价单挂着不一定成交。成交概率取决于：**前方排队量、主动成交流、价格是否被穿过、你撤不撤单**。排队位置由下单时间、价格、撤单速度决定：同样挂在买一，早挂 1 秒可能就排在几十手前面。排队越靠后，越可能“轮不到”。

### 4.2 逆向选择与 markout

passive 成交有个陷阱：你买在买价，成交后价格继续跌，说明对手方比你更急——这叫**逆向选择**。衡量方法是 **markout**：成交后一段时间（如 5 分钟）中间价相对成交价的变化。markout 为负，说明被动成交反而吃亏。

被动单省下半个价差，但可能输掉更多 markout；**只有在 markout 足够小、甚至为正时，做市/挂单才有正期望。**

### 4.3 代码：成交率与 markout



In [5]:

import numpy as np

rng = np.random.default_rng(5303)
n_orders = 2000
queue_ahead = rng.integers(100, 701, n_orders)          # 前方排队量
sell_flow = np.clip(rng.normal(430, 90, n_orders), 0, None)   # 1 分钟主动卖量
filled = sell_flow >= queue_ahead
fill_rate = filled.mean()

adverse = -0.0030 * (sell_flow - 430) / 430             # 教学设定：主动卖越多，价格越可能继续跌
markout = adverse + rng.normal(0, 0.0010, n_orders)

print(f"挂单 {n_orders} 笔（教学模拟），成交率 {fill_rate:.0%}，"
      f"成交单平均排队 {queue_ahead[filled].mean():.0f} 手")
print(f"成交单 5 分钟 markout 均值 {markout[filled].mean() * 1e4:+.2f}bp")
for lo, hi in [(100, 300), (300, 500), (500, 701)]:
    mask = (queue_ahead >= lo) & (queue_ahead < hi)
    sub = mask & filled
    if sub.sum() == 0:
        print(f"排队 {lo}-{hi} 手：样本 {mask.sum():>4}，无成交")
        continue
    print(f"排队 {lo}-{hi} 手：样本 {mask.sum():>4}，成交率 {filled[mask].mean():.0%}，"
          f"成交单 markout {markout[sub].mean() * 1e4:+.2f}bp")
print(f"被动买省下半个价差约 1.0bp，扣掉 markout 后净收益 "
      f"{(1.0 + markout[filled].mean() * 1e4):+.2f}bp（教学口径）")


挂单 2000 笔（教学模拟），成交率 56%，成交单平均排队 274 手
成交单 5 分钟 markout 均值 -1.46bp
排队 100-300 手：样本  694，成交率 98%，成交单 markout +0.14bp
排队 300-500 手：样本  674，成交率 59%，成交单 markout -3.27bp
排队 500-701 手：样本  632，成交率 7%，成交单 markout -10.02bp
被动买省下半个价差约 1.0bp，扣掉 markout 后净收益 -0.46bp（教学口径）



**读数与应用**：排队 100–300 手几乎全成交、markout 约 +0.1bp；排队 500 手以上成交率只有 7%，但成交单的 markout 是 −10bp——**一旦成交，往往正是主动卖压很大、价格还要继续跌的时候。** 整体被动买省下约 1bp 价差，markout −1.46bp，净 −0.46bp，扣手续费更差。被动交易的期望不是“省价差”，而是“省价差 − 逆向选择 − 手续费”。

**停下来想一想**：限价单成交率高，是不是就说明挂单策略好？

<details><summary>对照你的思路</summary>

不一定。成交率高可能只是因为价格挂得激进（甚至跨价成交），根本省不到价差；也可能是因为行情单边、成交后继续朝不利方向走。评价被动策略要同时看成交率、markout 和手续费：**成交率是“有多少”，markout 才是“好不好”。**

</details>

**常见错误**：

1. 回测假设限价单必成交，忽略队列位置。
2. 只算价差收益，不算 markout。
3. 在波动大、信息流强的时候挂单，被动成交后价格继续单边走。

---

## 5. 容量与执行对账

### 5.1 容量：给定成本预算能装多少

容量 = 在**可接受成本**下能执行的规模。它由三件事决定：市场成交量（ADV）、参与率上限、冲击成本曲线。规模越大，参与率越高，冲击越贵；规模小到一定地步，成本几乎等于半个价差，再小也没意义。用平方根模型粗算：容量 ∝ ADV × (成本预算/冲击参数)²，**成本预算翻倍，容量大约翻四倍**——所以容量对成本假设极其敏感。

### 5.2 执行对账：计划 vs 实际

对账不能只对“成交了多少手”，要对四件事：

1. **数量**：计划量、成交量、未成交量；
2. **价格**：到达价、成交 VWAP、滑点；
3. **成本**：价差、冲击、时机、手续费；
4. **差异**：实际成本与模型预测的残差，残差大说明模型漏了东西。

### 5.3 代码：一张执行对账表



In [6]:

import numpy as np

rng = np.random.default_rng(5304)
n_slices = 10
plan_qty = np.full(n_slices, 3000.0)
arrival = 100.00
mid = arrival + np.cumsum(rng.normal(0, 0.02, n_slices))


def impact_bps(q, adv=300000.0, c=0.6, sigma=0.02):
    return c * sigma * np.sqrt(q / adv) * 1e4


fill_qty = plan_qty.copy()
fill_qty[6] = 2400.0                                # 教学示例：有一片只成交 80%
fill_price = (mid + 0.01 + impact_bps(plan_qty) / 1e4 * arrival
              + rng.normal(0, 0.005, n_slices))

filled = fill_qty.sum()
actual_vwap = (fill_qty * fill_price).sum() / filled
slippage_bps = (actual_vwap - arrival) / arrival * 1e4
spread_bps = 0.01 / arrival * 1e4
impact_avg = (fill_qty * impact_bps(plan_qty)).sum() / filled
timing_bps = ((mid - arrival).mean() / arrival) * 1e4
residual = slippage_bps - (spread_bps + impact_avg + timing_bps)

rows = [
    ("计划量", f"{plan_qty.sum():,.0f} 手"),
    ("实际成交量", f"{filled:,.0f} 手"),
    ("未成交量", f"{plan_qty.sum() - filled:,.0f} 手"),
    ("到达价 / 成交VWAP", f"{arrival:.4f} / {actual_vwap:.4f}"),
    ("实际滑点", f"{slippage_bps:.2f}bp"),
    ("  价差部分", f"{spread_bps:.2f}bp"),
    ("  冲击部分", f"{impact_avg:.2f}bp"),
    ("  时机部分", f"{timing_bps:.2f}bp"),
    ("  残差（噪声）", f"{residual:+.2f}bp"),
    ("实际成本", f"{filled * (actual_vwap - arrival):,.0f} 元"),
    ("模型成本", f"{filled * arrival * (spread_bps + impact_avg + timing_bps) / 1e4:,.0f} 元"),
    ("未成交机会成本", f"{(plan_qty.sum() - filled) * (mid[-1] - arrival):,.0f} 元"),
]
for k, v in rows:
    print(f"{k:<16}{v:>18}")


计划量                       30,000 手
实际成交量                     29,400 手
未成交量                         600 手
到达价 / 成交VWAP    100.0000 / 100.0880
实际滑点                        8.80bp
  价差部分                      1.00bp
  冲击部分                     12.00bp
  时机部分                     -3.96bp
  残差（噪声）                   -0.24bp
实际成本                       2,587 元
模型成本                       2,659 元
未成交机会成本                      -26 元



**读数与应用**：成交 VWAP 高于到达价，滑点被拆成价差、冲击、时机三块，加总后与实际的差（残差）只剩噪声，说明成本模型没有漏项。未成交的 600 手要单独记机会成本——**对账表的最后一行不是“差异”，而是“还有哪些风险没算进去”。** 如果残差持续为正或持续为负，说明成本模型有系统性偏差，要先修模型再谈优化。

**常见错误**：

1. 对账只对成交量，不对价格。
2. 把滑点当成一个整体，不拆项，无法改进。
3. 未成交部分直接按 0 成本处理。

---

## 6. 从回测到实盘：执行假设清单

### 6.1 回测里最常见的四个执行假设

1. 用收盘价/中间价成交；
2. 流动性无限，想买多少买多少；
3. 零冲击、零滑点、零手续费；
4. 信号与成交同一根 K 线。

### 6.2 现实里对应发生什么

- 成交价要跨过半个价差；
- 大单吃穿盘口，冲击成本随规模上升；
- 限价单会排队、会不成交；
- 信号与成交之间至少差一个 bar，价格已经变了；
- 大单的冲击会反过来影响价格，你买得越多、成交越贵。

### 6.3 上实盘前的执行检查清单

- [ ] 回测的成交价是否含价差与冲击？
- [ ] 是否按品种校准过冲击系数 $c$？
- [ ] 是否有参与率上限与拆单规则？
- [ ] 限价单是否假设了成交率？
- [ ] 是否有未成交/部分成交的处理逻辑？
- [ ] 执行对账是否拆到价差/冲击/时机/残差？
- [ ] 容量是否用成本预算反推过？

---

## 本章小结

### 一张表

| 部件 | 输入 | 输出 | 典型失败 |
|---|---|---|---|
| 订单簿 | 盘口快照/逐笔 | 价差、深度、OFI | 把挂单量当可成交量 |
| 逐档吃量 | 母单量 + 盘口 | 成交均价、冲击 | 假设深度无限 |
| 执行算法 | 母单 + 成交量分布 | 拆单计划 | 只看均价不看未成交 |
| 成本模型 | 规模、波动、ADV | 冲击/总成本 | 线性外推 |
| 成交率模型 | 队列、主动流 | 成交率、markout | 假设必成交 |
| 对账 | 计划 + 成交明细 | 滑点分解、残差 | 只对数量不对价格 |

### 八句话

1. 中间价不是成交价，成交要跨过半个价差。
2. 盘口深度决定单笔能成交多少。
3. 大单的敌人是冲击，小单的敌人是价差。
4. TWAP 是基线，VWAP 跟随市场平均，POV 管参与率。
5. 比较执行算法要比“均价 + 未成交 + 成本”。
6. 冲击是次线性的，线性模型会两头出错。
7. 限价单成交率高不等于赚钱，要看 markout。
8. 执行对账要拆项，残差大说明模型漏了东西。

---

## 本章习题

**1.** 中间价 100 元、价差 2 跳（1 跳 0.01 元），立刻买入的“价差成本”是多少 bp？

**2.** 买盘 5 档共 3000 手，买入 4500 手，会发生什么？成交均价相对中间价是更好还是更差？

**3.** TWAP 和 VWAP 的核心区别是什么？什么情况下 VWAP 会明显优于 TWAP？

**4.** 某品种日成交量 20 万手、日波动 1.5%、冲击系数 0.6（教学示例）。下单 2 万手，平方根模型的冲击约多少 bp？

**5.** 成本预算 20bp、半个价差 1bp、$c\sigma=0.012$ 时，平方根模型的容量约多少（以 ADV 比例表示）？

**6.** 为什么限价单“成交了”反而可能亏钱？用什么指标衡量？

**7.** 执行对账要拆成哪几类成本？残差代表什么？

**8.** 回测里“按收盘价成交、无限流动性”会系统性高估什么？举两个受影响最大的策略类型。

<details><summary>参考答案</summary>

**1.** 半个价差 1 跳 $=0.01/100=1\text{bp}$。

**2.** 买盘深度只有 3000 手，4500 手会有 1500 手未成交（或需要等待/追价）；吃掉 5 档后成交均价高于中间价，越买越贵。

**3.** TWAP 按时间等分，VWAP 按成交量分布拆单。当日内成交量分布很不均匀（开盘/收盘放量）且价格与成交量相关时，VWAP 更贴近市场平均成本；没有成交量数据时 TWAP 更稳。

**4.** $0.6\times1.5\%\times\sqrt{20000/200000}\times10^4=0.6\times0.015\times0.3162\times10^4\approx28.5\text{bp}$。

**5.** 冲击预算 $=20-1=19\text{bp}$；$\sqrt{Q/V}=19/(0.012\times10^4)=0.1583$；$Q/V=0.0251$，即约 2.5% ADV。

**6.** 因为逆向选择：成交后价格继续朝不利方向走。用 markout（成交后一段时间中间价相对成交价的变化）衡量。

**7.** 价差、冲击、时机（执行期间价格漂移）、未成交机会成本（可单列）、手续费；残差是实际成本与模型成本之差，代表模型未解释的部分（噪声或漏项）。

**8.** 高估策略容量与净收益，低估换手成本；受影响最大的是高换手策略（日内/高频）和大资金策略（规模已经接近容量上限）。

</details>

---

## 参考资料

- Almgren, R. & Chriss, N. (2001). *Optimal Execution of Portfolio Transactions*.
- Kissell, R. *The Science of Algorithmic Trading and Portfolio Management*.
- O'Hara, M. *Market Microstructure Theory*。
- 第 14、31 章；第 03 章（因子报告中的换手与成本）。
